# Entrenamiento en Google Colab

Entorno → Cambiar tipo de entorno de ejecución → **GPU T4**.

Flujo:
1. Montar Drive y clonar el repositorio.
2. Copiar los rostros ya extraídos (`.zip` en Drive) al disco local de Colab — leer miles de imágenes directo desde Drive es muy lento.
3. Entrenar guardando checkpoints en Drive. Si la sesión se cae, se vuelve a correr todo y la celda de entrenamiento continúa con `--reanudar`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Carpeta del proyecto en Drive (ajustar si es necesario)
DRIVE = '/content/drive/MyDrive/deepfakes'
!mkdir -p {DRIVE}/runs {DRIVE}/results

## Código

Si el repositorio es privado, crea un *token* de GitHub (solo lectura) y guárdalo en los **Secretos** de Colab (icono de llave) con el nombre `GITHUB_TOKEN`.

In [ ]:
import os
REPO = 'Julianchu850/proyecto-redes-neuronales-deteccion-deepfake'
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    url = f'https://{token}@github.com/{REPO}.git'
except Exception:
    url = f'https://github.com/{REPO}.git'

if not os.path.exists('/content/repo'):
    !git clone -q {url} /content/repo
else:
    !git -C /content/repo pull -q
%cd /content/repo
!pip install -q -r requirements.txt

## Datos

Se espera en Drive:
- `deepfakes/faces_ffpp.zip`: la carpeta de rostros generada por `src.data.extraer_rostros` (incluye `manifiesto.csv`).
- Los splits ya están versionados en el repositorio (`data/splits/ffpp`). Si no, se generan aquí.

In [ ]:
!mkdir -p /content/data/faces
!unzip -q -n {DRIVE}/faces_ffpp.zip -d /content/data/faces/
!ls /content/data/faces

In [ ]:
import os
if not os.path.exists('data/splits/ffpp/train.csv'):
    !python -m src.data.crear_splits --dataset ffpp --rostros /content/data/faces/ffpp --salida data/splits/ffpp

## Entrenamiento — línea base MesoNet (semana 1)

In [ ]:
MODELO = 'meso4'
RUN = f'{DRIVE}/runs/{MODELO}'
!python -m src.entrenar --modelo {MODELO} --datos /content/data/faces/ffpp --splits data/splits/ffpp \
    --salida {RUN} --epocas 30 --batch 64 --workers 2 --reanudar

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
h = pd.read_csv(f'{RUN}/historial.csv')
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(h.epoca, h.perdida_train); ax[0].set_title('Pérdida de entrenamiento'); ax[0].set_xlabel('época')
ax[1].plot(h.epoca, h.auc_val_frame, label='frame'); ax[1].plot(h.epoca, h.auc_val_video, label='video')
ax[1].set_title('AUC de validación'); ax[1].set_xlabel('época'); ax[1].legend()
plt.tight_layout()

## Evaluación en el conjunto de prueba (E1)

In [ ]:
!python -m src.evaluar --checkpoint {RUN}/mejor.pt --datos /content/data/faces/ffpp \
    --csv data/splits/ffpp/test.csv --salida {DRIVE}/results/{MODELO}_ffpp_test